# BTC + ETH Options — Volatility Risk Premium Strategy

**Personal Quantitative Research Project**  
*MSc Quantitative Finance · Singapore Management University*

---

Systematic short-volatility strategy on Deribit BTC + ETH weekly options. Walk-forward LightGBM gating + vol-target sizing + full-friction path simulation.

This notebook shows **results only**. Source code, math, and audits are in the [README](../README.md) and the source repo.

> **Disclaimer:** Research-only. Not investment advice. Hypothetical backtest; live results may differ.

## 1. The Leak Story

Initial backtest reported Sharpe ~9.83 with max drawdown ~1%. Suspicious. Audit found a 16-hour HAR forecast date-join leak (daily forecast applied at intraday Friday 08:00 entry). After fixing:

| Stage | Sharpe | Note |
|---|---:|---|
| Original (with leak) | 9.83 | Too good — was too good |
| 1-day-lagged stress test | 0.72 | Confirms leak: ΔSharpe = +9.1 is the leak signal |
| Leak-free + vol-target sizing | **3.73 BTC / 4.28 ETH** | Final defensible result |

**Lesson:** when a backtest looks too clean, audit the date joins.

![Leak fix](../figures/lookahead_audit.png)

## 2. BTC 2025 — Out-of-Sample Results

Strategy trained on data pre-2025 only. Evaluated fresh on 2025.

| Variant | Trades 2025 | Sharpe 2025 | Cum 2025 ($1M) | Hit |
|---|---:|---:|---:|---:|
| Hard rule (vrp_z>1.0) | 5 | -0.17 | -$1k | failed |
| XGBoost ensemble | 5 | 7.25 | $27k | small N |
| **LightGBM rolling 12m / 3m** ⭐ | **37** | **3.73** | **$154k** | 62% |

Hard rule failed in 2025 because BTC's VRP regime compressed 62% — static thresholds calibrated on 2022-2024 fired on the wrong weeks. Adaptive ML (rolling LightGBM) recovered most of the lost edge by retraining every 13 weeks.

![BTC 2025 equity](../figures/btc_2025_voltarget.png)

## 3. ETH 2025 — Cross-Asset Replication

Same code applied to ETH (15.3M Deribit option ticks downloaded fresh, 444 MB). Strongest possible OOS test — different asset, no parameter changes.

| Variant | Trades 2025 | Sharpe 2025 | Cum 2025 ($1M) | Hit |
|---|---:|---:|---:|---:|
| Hard rule (vrp_z>1.0) | 10 | **5.23** | $105k | 74% |
| Hard rule (loose) | 20 | 3.59 | $152k | 68% |
| **LightGBM rolling 12m / 3m** ⭐ | **25** | **4.28** | **$238k** | 70% |
| LightGBM 24m / 3m | 32 | 4.04 | $286k | 67% |

Hard rules **work** on ETH 2025 (where they failed on BTC), confirming the BTC failure was a regime issue specific to BTC's 2025 VRP compression — not a strategy flaw.

![ETH 2025 monthly](../figures/eth_2025_monthly.png)

## 4. ML Model Comparison — XGBoost vs LightGBM

LightGBM rolling 12m train / 3m test beats XGBoost expanding-window on both assets:

| | BTC 2025 Sharpe | BTC 2025 Cum | ETH 2025 Sharpe | ETH 2025 Cum |
|---|---:|---:|---:|---:|
| XGBoost expanding | 2.59 | $115k | 2.84 | $229k |
| **LightGBM rolling** | **3.73** | **$154k** | **4.28** | **$238k** |
| Lift | +44% Sharpe | +34% PnL | +51% Sharpe | +4% PnL |

Rolling adapts to regime shifts (rolls old data out) while still having enough samples (52 weeks) for trees to fit non-linear patterns.

![BTC LightGBM](../figures/btc_lightgbm_rolling.png)

![ETH LightGBM](../figures/eth_lightgbm_rolling.png)

## 5. Is the ML Edge Real or Random?

**Test:** at the same trade count, sample 5000 random gates from the full population. Where does ML's Sharpe sit in the random distribution?

| Asset | Period | n trades | ML Sharpe | Random p50 | Random p95 | **ML percentile** | Verdict |
|---|---|---:|---:|---:|---:|---:|---|
| BTC | full | 107 | 3.07 | 3.31 | 4.11 | 30% | random ≈ ML |
| BTC | 2025 | 37 | 3.73 | 2.81 | 4.48 | 83% | borderline |
| ETH | full | 98 | 3.85 | 3.28 | 4.13 | 86% | borderline |
| **ETH** | **2025** | **25** | **4.28** | 2.11 | 3.85 | **97.5%** | **real edge** |

**ETH 2025 ML signal is genuine** — only 2.5% chance the result is from random luck.

**BTC ML signal is marginal** — most of BTC's headline comes from baseline VRP + vol-target sizing, not ML selection. Honest finding: BTC weekly VRP is structurally positive, so random selection already does well.

![ML vs random](../figures/ml_vs_random.png)

## 6. Risk Limits — Live Deploy Stress Test

Five risk limits added (notional cap, funding-spike skip, 4w drawdown halt, consecutive-loss halt, 1w loss skip). In baseline the limits don't trigger (strategy is well-behaved). Under loss-amplification stress they activate and save real money:

| Loss × | Cum PnL no-limits | Cum PnL w/limits | Savings |
|---|---:|---:|---:|
| 1.0× (baseline) | +$316k | +$270k | -$45k |
| 2.0× (mild) | +$165k | +$130k | -$35k |
| **3.0× (moderate)** | +$14k | **+$64k** | **+$49k** |
| **5.0× (LUNA-tier)** | -$287k | **-$108k** | **+$180k** |

Limits act as insurance: small cost in normal regime, big payoff in fat tails.

![ETH risk limits](../figures/eth_risk_limits.png)

## 7. Summary

| Metric | BTC 2025 | ETH 2025 |
|---|---:|---:|
| Trades | 37 | 25 |
| Cumulative PnL | +$154k | +$238k |
| Total return on $1M | 15.4% | 23.8% |
| Sharpe (52w ann) | 3.73 | 4.28 |
| Hit rate | 62% | 70% |
| Max month DD | -2.4% | -1.3% |
| ML vs random percentile | 83% | **97.5%** |

### What this project shows
1. End-to-end systematic strategy on raw exchange ticks (BTC + ETH)
2. Found + fixed a real look-ahead leak — Sharpe corrected from 9.8 to 3.7-4.3
3. LightGBM rolling beats XGBoost expanding by 44-51% Sharpe lift
4. Cross-asset replication (BTC → ETH) — strategy generalizes
5. Honest reporting: ML vs random Monte Carlo passes only on ETH; BTC ML edge marginal

### What this project does NOT claim
- Sharpe 9 (that was the leaky number — discarded)
- Profitable in all conditions (BTC 2025 hard rule failed)
- Production-ready (paper-trade required first)

---

**Karan Chavan** · MSc Quantitative Finance, Singapore Management University · [karan80121@gmail.com](mailto:karan80121@gmail.com)